In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import numpy as np


def broken_powerlaw_normalization(
    rho_iso,
    rho_bulk,
    q1,
    q2,
    r_min,
    r_break,
    r_max,
):
    """
    Normalize a broken power-law ISO size distribution from a mass density.

    dn/dr = A1 * r^(-q1),   r_min <= r < r_break
          = A2 * r^(-q2),   r_break <= r <= r_max

    Continuity at r_break requires:
        A2 = A1 * r_break^(q2 - q1)

    Parameters
    ----------
    rho_iso : float
        Total ISO mass density.
        Example: g / pc^3

    rho_bulk : float
        Bulk density of individual objects.
        Example: g / cm^3

    q1, q2 : float
        Differential size-distribution slopes.

    r_min, r_break, r_max : float
        Object radii. These must all use the same length unit.
        Example: cm

    Returns
    -------
    A1, A2 : floats
        Normalizations of the two branches.
    """

    # ----- Small-object contribution to the MASS integral -----

    if np.isclose(q1, 4.0):
        mass_term_1 = np.log(r_break / r_min)

    else:
        mass_term_1 = (
            r_break**(4.0 - q1)
            - r_min**(4.0 - q1)
        ) / (4.0 - q1)


    # ----- Large-object contribution to the MASS integral -----

    continuity_factor = r_break**(q2 - q1)

    if np.isclose(q2, 4.0):
        mass_term_2 = (
            continuity_factor
            * np.log(r_max / r_break)
        )

    else:
        mass_term_2 = (
            continuity_factor
            * (
                r_max**(4.0 - q2)
                - r_break**(4.0 - q2)
            )
            / (4.0 - q2)
        )


    # ----- Combine the two mass contributions -----

    total_mass_integral = mass_term_1 + mass_term_2


    # rho_iso = (4*pi/3) * rho_bulk * A1 * total_mass_integral

    A1 = rho_iso / (
        (4.0 * np.pi / 3.0)
        * rho_bulk
        * total_mass_integral
    )


    # Continuity at the break

    A2 = A1 * continuity_factor

    return A1, A2

In [3]:
def integrate_number_powerlaw(A, q, r_low, r_high):
    """
    Integrate A * r^(-q) from r_low to r_high.
    """

    if np.isclose(q, 1.0):
        return A * np.log(r_high / r_low)

    return (
        A
        * (
            r_high**(1.0 - q)
            - r_low**(1.0 - q)
        )
        / (1.0 - q)
    )


def number_density_in_bin(
    rho_iso,
    rho_bulk,
    q1,
    q2,
    r_min,
    r_break,
    r_max,
    R1,
    R2,
):
    """
    Number density of ISOs with radii R1 < r < R2.

    The size distribution is normalized using rho_iso.
    """

    if R1 < r_min or R2 > r_max:
        raise ValueError(
            "Requested size bin must lie inside "
            "[r_min, r_max]."
        )

    if R1 >= R2:
        raise ValueError("R1 must be smaller than R2.")


    A1, A2 = broken_powerlaw_normalization(
        rho_iso=rho_iso,
        rho_bulk=rho_bulk,
        q1=q1,
        q2=q2,
        r_min=r_min,
        r_break=r_break,
        r_max=r_max,
    )


    # Entire interval below the break
    if R2 <= r_break:

        N = integrate_number_powerlaw(
            A=A1,
            q=q1,
            r_low=R1,
            r_high=R2,
        )


    # Entire interval above the break
    elif R1 >= r_break:

        N = integrate_number_powerlaw(
            A=A2,
            q=q2,
            r_low=R1,
            r_high=R2,
        )


    # Interval crosses the break
    else:

        N_below = integrate_number_powerlaw(
            A=A1,
            q=q1,
            r_low=R1,
            r_high=r_break,
        )

        N_above = integrate_number_powerlaw(
            A=A2,
            q=q2,
            r_low=r_break,
            r_high=R2,
        )

        N = N_below + N_above


    return N

In [6]:
AU = 1.495978707e13
km = 1.0e5
m  = 100.0
cm = 1.0
M_Earth=5.972e+27

rho_iso  = M_Earth    # g / pc^3
rho_bulk = 1.0       # g / cm^3

q1 = 3.0
q2 = 4.0

r_min   = 1.0e-4 * cm      # 1 micron
r_break = 30.0 * km
r_max   = 1000.0 * km

R1 = 50.0 * m
R2 = 500.0 * m

N = number_density_in_bin(
    rho_iso=rho_iso,
    rho_bulk=rho_bulk,
    q1=q1,
    q2=q2,
    r_min=r_min,
    r_break=r_break,
    r_max=r_max,
    R1=R1,
    R2=R2,
)

print(f"N = {N:.3e} pc^-3")

N = 2.088e+12 pc^-3


In [9]:
N_gt_100m = number_density_in_bin(
    rho_iso=rho_iso,
    rho_bulk=rho_bulk,
    q1=q1,
    q2=q2,
    r_min=r_min,
    r_break=r_break,
    r_max=r_max,
    R1=100.0 * m,
    R2=r_max,
)
print(f"{N_gt_100m:e}")

5.272704e+11


527270380934.97943